# Week 8 (Tue) - Class Activities in CS 315

You are given the file `netflix_top10_countries.tsv` in the folder data, solve the following tasks using it.

**Table of Contents**

1. [Q1: Load the file](#q1)
2. [Q2: Show number of rows](#q2)
3. [Q3: Show column names](#q3)
4. [Q4: Show number of total countries](#q4)
5. [Q5: Show number of TV series](#q5)
6. [Q6: Number of TV series in South Korea](#q6)
7. [Q7: Total weeks of Squid Games in Italy ](#q7)
8. [Q8: Time frame for a given country](#q8)
9. [Q9: Is number of weeks in ranking equal?](#q9)
10. [Q10: Country with fewest weeks](#q10)


<a id="q1"></a>
### Q1: Load the file

In [2]:
%pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 8.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 4.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
df = pd.read_csv("all-weeks-countries.tsv", sep="\t")


<a id="q2"></a>
### Q2: Show number of rows

In [4]:
len(df)

515920

<a id="q3"></a>
### Q3: Show column names

In [5]:
df.columns


Index(['country_name', 'country_iso2', 'week', 'category', 'weekly_rank',
       'show_title', 'season_title', 'cumulative_weeks_in_top_10'],
      dtype='str')

<a id="q4"></a>
### Q4: Show number of total countries

In [6]:
df["country_name"].nunique()


94

<a id="q5"></a>
### Q5: Show number of TV series

In [7]:
df[df["category"] == "TV"]["show_title"].nunique()


3607

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [8]:
df[(df["category"] == "TV") & (df["country_name"] == "South Korea")]["show_title"].nunique()


486

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [9]:
df[(df["show_title"] == "Squid Game") & (df["country_name"] == "Italy")]["week"].nunique()


29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [10]:
def show_timeframe(df, country):
    weeks = df[df["country_name"] == country]["week"]
    print(f"Netflix rankings for {country} exist between {weeks.min()} and {weeks.max()}.")
show_timeframe(df, "Chile")
show_timeframe(df, "Russia")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-18.
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

The total number of weeks in the rankings is `len(df['week'].unique())`.

The total number of countries is `len(df['country_name'].unique())`.  

For every week, each country has 20 rows.   

If they all stayed the same number of weeks, then the product of these three weeks would be the same as the total number of rows.  

In [11]:
df.groupby("country_name")["week"].nunique().nunique() == 1


False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Write a one-liner that will show the country with the fewest weeks in the Netflix rankings

In [12]:
df.groupby("country_name")["week"].nunique().reset_index(name="weekCnt").sort_values("weekCnt").head()


,country_name,weekCnt
70,Russia,35
67,Portugal,277
66,Poland,277
65,Philippines,277
64,Peru,277


## Task 2 New Task part

In [13]:
#rank2score: F1 points for 1st through 10th place
def rank2score(rank):
    points = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
    return points.get(rank, 0)

#New score column, without the columns that aren't useful
df["score"] = df["weekly_rank"].apply(rank2score)
df2 = df.drop(columns=["country_iso2", "season_title", "cumulative_weeks_in_top_10"])
df2.head()

#Lovely Runner: which countries it did well in
(df2[df2["show_title"] == "Lovely Runner"]
    .groupby("country_name")
    .agg(weeks=("week", "nunique"),
         bestRank=("weekly_rank", "min"),
         totalScore=("score", "sum"))
    .reset_index()
    .sort_values("totalScore", ascending=False))

,country_name,weeks,bestRank,totalScore
12,Vietnam,7,1,75
2,Indonesia,7,1,69
3,Malaysia,6,1,62
6,Philippines,4,3,41
4,Maldives,4,3,32
9,Sri Lanka,4,3,32
0,Bangladesh,3,4,30
1,India,3,4,26
7,Singapore,4,5,20
11,Thailand,3,5,20


In [14]:
def show_success(df, title):
    return (df[df["show_title"] == title]
            .groupby("country_name")
            .agg(weeks=("week", "nunique"),
                 bestRank=("weekly_rank", "min"),
                 totalScore=("score", "sum"))
            .reset_index()
            .sort_values("totalScore", ascending=False))

#Look up the exact title
df2[df2["show_title"].str.contains("Tangerines", case=False)]["show_title"].unique()

for title in ["Squid Game", "Extraordinary Attorney Woo", "The Glory",
              "Queen of Tears", "When Life Gives You Tangerines", "Business Proposal"]:
    print(title)
    display(show_success(df2, title).head(10))

Squid Game


,country_name,weeks,bestRank,totalScore
33,India,51,1,816
61,Pakistan,45,1,808
5,Bangladesh,48,1,807
40,Jordan,36,1,772
7,Bolivia,37,1,760
42,Kuwait,34,1,738
9,Bulgaria,36,1,738
20,Egypt,35,1,724
60,Oman,32,1,705
86,Turkey,37,1,702


Extraordinary Attorney Woo


,country_name,weeks,bestRank,totalScore
22,Japan,25,1,370
49,Taiwan,17,1,308
18,Hong Kong,18,1,295
45,Singapore,17,1,284
46,South Korea,16,1,268
20,Indonesia,15,1,262
56,Vietnam,13,1,239
39,Peru,16,1,236
50,Thailand,24,1,234
27,Malaysia,15,1,231


The Glory


...

,country_name,weeks,bestRank,totalScore
45,Vietnam,17,1,294
40,Taiwan,15,1,267
17,Indonesia,14,1,260
33,Philippines,17,1,248
15,Hong Kong,14,1,231
21,Malaysia,13,1,229
38,South Korea,13,1,220
37,Singapore,13,1,212
41,Thailand,13,1,194
26,Nicaragua,11,1,173


When Life Gives You Tangerines


,country_name,weeks,bestRank,totalScore
2,Bangladesh,21,2,219
37,Taiwan,11,1,192
35,South Korea,14,1,185
42,Vietnam,10,1,170
16,Indonesia,9,1,166
31,Philippines,9,1,161
30,Peru,12,1,158
15,India,14,2,156
14,Hong Kong,10,1,147
20,Malaysia,10,1,140


Business Proposal


,country_name,weeks,bestRank,totalScore
19,Indonesia,21,1,264
2,Bangladesh,29,1,256
26,Malaysia,14,1,226
57,Vietnam,13,1,225
41,Philippines,13,1,221
52,Thailand,18,1,216
17,Hong Kong,12,1,211
21,Japan,12,1,210
47,Singapore,13,1,198
51,Taiwan,12,1,194
